# 3. Classify

Runs the configured classification strategy for every active schema.
Each schema declares its strategy in `SchemaSpec.classify_strategy`:

| Strategy | When | How |
|----------|------|-----|
| `ai_classify` | leaf_count ≤ 300 | Hierarchical SQL `ai_classify` walks the schema's `level_columns`. Each level sees ≤20 candidates (the primitive's cap). |
| `retrieval_llm` | leaf_count > 300 | Per invoice: tsvector top-K candidates → `ai_query` STRUCT prompt → model picks. Uses the index built in notebook 2. |
| `tsvector` | optional baseline | Top-1 from the tsvector index, no LLM. Cheap reference for benchmarking. |

All three write into the unified `cat_predictions` table with a clear
`source` label and a confidence normalized via `src.confidence`.

**Adding a new schema?** Subclass `ClassificationSchema`, set
`classify_strategy` on its spec, register it in `src/schemas/__init__.py`.
Re-run notebook 0 to write the taxonomy table, notebook 2 to index it,
then this notebook. No edits required here.

Tested on Serverless v4.

In [ ]:
%pip install pyyaml openai openpyxl pydantic 'databricks-sdk>=0.50.0' psycopg2-binary
%restart_python

In [ ]:
import os, sys, uuid
sys.path.insert(0, os.path.abspath('..'))

from src.utils import get_spark
from src.config import load_config
from src.schemas import list_schemas, get_schema
from src.categorize import (
    classify_ai_classify,
    classify_retrieval_llm,
    classify_tsvector,
    run_strategy_for,
)

spark = get_spark()
config = load_config()

In [ ]:
dbutils.widgets.removeAll()
dbutils.widgets.text('catalog', config.catalog)
dbutils.widgets.text('schema', config.schema_name)
dbutils.widgets.text('invoices_table', config.invoices)
dbutils.widgets.text('lakebase_instance', config.lakebase_instance)
dbutils.widgets.text('llm_endpoint', config.large_llm_endpoint)
dbutils.widgets.text('limit_rows', '0')   # 0 = all
dbutils.widgets.text('also_run_tsvector', 'false')   # baseline alongside primary

catalog = dbutils.widgets.get('catalog')
schema = dbutils.widgets.get('schema')
invoices_table = dbutils.widgets.get('invoices_table')
lakebase_instance = dbutils.widgets.get('lakebase_instance')
llm_endpoint = dbutils.widgets.get('llm_endpoint')
limit_rows = int(dbutils.widgets.get('limit_rows'))
also_run_tsvector = dbutils.widgets.get('also_run_tsvector').lower() == 'true'
spark.sql(f'USE {catalog}.{schema}')

## Lakebase connection

`retrieval_llm` and `tsvector` strategies query the Postgres tsvector
indexes built in notebook 2. We pass a fresh-connection factory so each
worker in the parallel `retrieval_llm` path can hold its own connection.

In [ ]:
from databricks.sdk import WorkspaceClient
import psycopg2

w = WorkspaceClient()

def pg_connect():
    inst = w.database.get_database_instance(name=lakebase_instance)
    cred = w.database.generate_database_credential(
        request_id=str(uuid.uuid4()), instance_names=[lakebase_instance]
    )
    me = w.current_user.me()
    user = me.emails[0].value if me.emails else me.user_name
    return psycopg2.connect(
        host=inst.read_write_dns, dbname=config.lakebase_dbname,
        user=user, password=cred.token, sslmode='require',
    )

## Run the configured strategy for every schema

`run_strategy_for` looks at `schema.spec.classify_strategy` and dispatches
to the right function in `src.categorize`. Each strategy MERGEs results
into `cat_predictions` with a clear source label.

In [ ]:
for s in list_schemas():
    print(f'\n==> {s.name} ({s.spec.classify_strategy})')
    source, n = run_strategy_for(
        spark, s,
        catalog=catalog, schema_db=schema,
        invoices_table=invoices_table,
        pg_connect=pg_connect,
        llm_endpoint=llm_endpoint,
        limit_rows=limit_rows,
    )
    print(f'    merged {n:,} rows with source={source}')

## Optional: run tsvector baseline alongside the primary strategy

Set the `also_run_tsvector` widget to `true` to add a `source='tsvector'`
row for every schema. Useful for benchmarking the LLM strategies against
a free baseline. Off by default.

In [ ]:
if also_run_tsvector:
    for s in list_schemas():
        if s.spec.classify_strategy == 'tsvector':
            continue  # already covered above
        print(f'\n==> {s.name} (tsvector baseline)')
        n = classify_tsvector(
            spark, s,
            catalog=catalog, schema_db=schema,
            invoices_table=invoices_table,
            pg_connect=pg_connect,
        )
        print(f'    merged {n:,} tsvector rows')
else:
    print('Skipping tsvector baseline. Flip widget `also_run_tsvector` to true to enable.')

## Sanity check

In [ ]:
spark.sql(f'''
SELECT schema_name, source,
       COUNT(*) AS n,
       COUNT(confidence) AS with_conf,
       ROUND(AVG(confidence), 3) AS avg_conf
FROM {catalog}.{schema}.cat_predictions
GROUP BY schema_name, source
ORDER BY schema_name, source
''').display()